In [10]:
#importlibraries

#libraries to extract frame
import os
import pandas as pd
import cv2
import numpy as np

#libraries to undergo differencing
#import torch
#from torchvision import transforms
from PIL import Image
#import numpy as np
import matplotlib.pyplot as plt
#from torchvision.utils import save_image


#vid_locations path
vid_locations_path = "../../data/Deepfish/location_data/vid_locations.csv"
#vid_locations_path = "../../data/Tassy_BRUV/location_data/vid_locations.csv"

#path where augmented videos are to be saved
aug_video_path = "../../data/Deepfish/augmented_videos/background_subtraction/"
#aug_video_path = "../../data/Tassy_BRUV/augmented_videos/background_subtraction/"

#image size
image_size = [1920,1080]

#kernel for open/close morphology operations (set to False if not used)
kernel = False

#here are some example kernels:
        #kernel = np.ones((5,5),np.uint8)
    
        #kernel = np.array([[0, 0, 1, 0, 0],
        #   [0, 1, 1, 1, 0],
        #   [1, 1, 1, 1, 1],
        #   [0, 1, 1, 1, 0],
        #   [0, 0, 1, 0, 0]], dtype=np.uint8)

#whether to use 'open' morphological tranformation (needs a kernel)
open = False

#whether to use 'close' morphological tranformation (needs a kernel)
close = False


In [27]:
#using KNN
vid_locations_dat = pd.read_csv(vid_locations_path)


for i in range(len(vid_locations_dat["video_location"])):
    cap = cv2.VideoCapture(vid_locations_dat["video_location"][i])
    
    fgbg = cv2.createBackgroundSubtractorKNN(detectShadows = False)

    height, width, layers = (image_size[1], image_size[0], 1)

    #write the video
    fourcc = cv2.VideoWriter_fourcc('M','J','P','G') 
    video = cv2.VideoWriter(aug_video_path + vid_locations_dat["video_name"][i], fourcc, 10, (width,height),isColor = False)

    while(1): 
        ret, frame = cap.read()
        if frame is None:
            break

        fgmask = fgbg.apply(frame) 
        
        if(kernel!=False):
            if(open!=False):
                fgmask = cv2.morphologyEx(fgmask, cv2.MORPH_OPEN, kernel)
            if(close!=False):
                fgmask = cv2.morphologyEx(fgmask, cv2.MORPH_CLOSE, kernel)
                
        video.write(fgmask)

In [13]:
i=11
cap = cv2.VideoCapture(vid_locations_dat["video_location"][i])

fgbg = cv2.createBackgroundSubtractorKNN(detectShadows = False)

height, width, layers = (image_size[1], image_size[0], 1)

#write the video
fourcc = cv2.VideoWriter_fourcc('M','J','P','G') 
#fourcc = cv2.VideoWriter_fourcc(*'XVID') 
video = cv2.VideoWriter(aug_video_path + vid_locations_dat["video_name"][i], fourcc, 60, (width,height),isColor = False)

while(1): 
    ret, frame = cap.read()
    if frame is None:
        break

    fgmask = fgbg.apply(frame) 
    
    if(kernel!=False):
        if(open!=False):
            fgmask = cv2.morphologyEx(fgmask, cv2.MORPH_OPEN, kernel)
        if(close!=False):
            fgmask = cv2.morphologyEx(fgmask, cv2.MORPH_CLOSE, kernel)
            
    video.write(fgmask)

In [31]:
i=11
target_frame = 300
lead_in = 100

cap = cv2.VideoCapture(vid_locations_dat["video_location"][i])

cap.set(cv2.CAP_PROP_POS_FRAMES, max(0,target_frame-lead_in))

fgbg = cv2.createBackgroundSubtractorKNN(detectShadows = False,history=lead_in)

height, width, layers = (image_size[1], image_size[0], 1)

#write the video
#fourcc = cv2.VideoWriter_fourcc('M','J','P','G') 
#fourcc = cv2.VideoWriter_fourcc(*'XVID') 
#video = cv2.VideoWriter(aug_video_path + vid_locations_dat["video_name"][i], fourcc, 60, (width,height),isColor = False)

while(1): 
    ret, frame = cap.read()
    if frame is None:
        break

    fgmask = fgbg.apply(frame) 
    
    if(kernel!=False):
        if(open!=False):
            fgmask = cv2.morphologyEx(fgmask, cv2.MORPH_OPEN, kernel)
        if(close!=False):
            fgmask = cv2.morphologyEx(fgmask, cv2.MORPH_CLOSE, kernel)
    
    frame_number = cap.get(cv2.CAP_PROP_POS_FRAMES)
    if(frame_number == target_frame):
        im = Image.fromarray(fgmask)
        im.save("../../data/Deepfish/augmented_videos/background_subtraction/test.jpg")
        break
            
    #video.write(fgmask)

In [ ]:
#4m 56.3s with 10 MJPG

#2m 24.5s with 50 MJPG (more frames from 60 is bad)

#3m 11.3s with 60 *'XVID' 

#2m 52.2s with 50 MJPG


#frame in above code is frame number, you could let it run a few hundred frames before the frame you are interested in and then just extract out the one frame you want